## Comparing 2 LLM embedding models (both suitable for Norwegian text)

The original plan was to compare a large and small model, but larger models are hard to run without GPU. We could be able to do this eventually when working on Colab etc.

This file:
- Compare tf-idf, bm25, 2 llm embedding models, and 2 vector comparison methods (euclidean and cosine). Cosine is likely to work better, but we have more to compare.

In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent

from backend.ai.similarity_metrics.shared_logic import load_corpus
from backend.ai.similarity_metrics.bm25_similarity import bm25_similarity
from backend.ai.similarity_metrics.tfidf_similarity import tf_idf_similarity
from backend.ai.similarity_metrics.llm_embedding_similarity import llm_embedding_similarity

/home/lucas/miniconda3/envs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
SMALL_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
BIG_MODEL = "NbAiLab/borealis-embed-212m"

In [5]:
def get_similarity_scores(questions_path = "data/raw/serviceNow/serviceNow_questions.xlsx",
         routines_path = "data/raw/kvaliteket"):
    """
    Reads the PDFs and builds the corpus once, then runs all three similarity
    methods on that same corpus, asserts each result has shape
    questions x routines, and prints the top match per question for each
    method as a sanity check.
    """

    scores_dict = {}

    corpus = load_corpus(questions_path, routines_path)
    num_questions = len(corpus.questions)
    num_routines = len(corpus.routine_names)

    methods = {
        "tf-idf": tf_idf_similarity,
        "bm25": bm25_similarity,
        # We consider the default model small
        "cosine_small": lambda corpus: llm_embedding_similarity(corpus, model_name=SMALL_MODEL, similarity_metric="cosine"),
        "cosine_large": lambda corpus: llm_embedding_similarity(corpus, model_name=BIG_MODEL, similarity_metric="cosine"),
        "euclidean_small": lambda corpus: llm_embedding_similarity(corpus, model_name=SMALL_MODEL, similarity_metric="euclidean"),
        "euclidean_large": lambda corpus: llm_embedding_similarity(corpus, model_name=BIG_MODEL, similarity_metric="euclidean")
    }

    for name, method in methods.items():
        scores = method(corpus)
        scores_dict[name] = scores

    return scores_dict

In [6]:
questions_path = REPO_ROOT / "data/raw/serviceNow/serviceNow_questions.xlsx"
routines_path = REPO_ROOT / "data/raw/kvaliteket"

scores_dict = get_similarity_scores(questions_path=questions_path, routines_path=routines_path)

  - 5073-1-Lokal særavtale_Kulturskolen_Private instrumenter.pdf
  - 5079-1-Lokal avtale_honorering av fastleger som deltar i kommunale møter.pdf
  - 5080-1-Lokal avtale_Gjennomsnittsberegning av arbeidstiden for ansatte i barnehagene.pdf
  - 5081-1-Lokal avtale_Arbeidstid for avdelingsledere for turnusbaserte tjenester.pdf
  - 5083-1-Lokal særavtale_Arbeidstid ved Trondheim kommunale kulturskole.pdf
  - 5087-1-Lokal særavtale_Arbeidstøy.pdf
  - 5088-1-Lokal særavtale_Planleggingstid i barnehage og sfo for assistenter og fagarbeider.pdf
  - 5091-1-Lokal avtale_kortere daglig arbeidsfri og kortere ukehvile for vikarer, ved merarbeid og ønske om å bytte vakt.pdf
  - 9033-1-Lokal avtale_Arbeidstid avdelingsledere i barnehage.pdf
  - 9237-1-Lokal avtale_Arbeidstid avdelingsledere i skole.pdf


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 2645.72it/s]


In [12]:
import numpy as np
import pandas as pd


def clean(s, name):
    s = np.asarray(s, dtype=float)
    return np.where(np.isnan(s), -np.inf, s)

def top_doc(scores, name):
    s = clean(scores, name)
    return s.argmax(axis=1)

def diagnose(scores_dict, true_docs=None):
    """
    scores_dict: {method: array (num_questions, num_documents)}
    true_docs:   optional array (num_questions,) with the correct doc index
    """
    preds, rows = {}, []
    for name, s in scores_dict.items():
        s = np.asarray(s, dtype=float)
        p = top_doc(s, name)
        preds[name] = p
        counts = np.bincount(p, minlength=s.shape[1])

        srt = np.sort(s, axis=1)
        margin = (srt[:, -1] - srt[:, -2])

        row = {
            "method": name,
            "shape": s.shape,
            "nan_count": int(np.isnan(s).sum()),
            "unique_docs_predicted": int((counts > 0).sum()),
            "top_doc": int(counts.argmax()),
            "top_doc_share": round(counts.max() / len(p), 3),
            "flat_rows": int((np.ptp(s, axis=1) < 1e-9).sum()),
            "mean_margin": round(float(np.nanmean(margin)), 4),
        }
        if true_docs is not None:
            row["accuracy@1"] = round(float((p == np.asarray(true_docs)).mean()), 3)
        rows.append(row)

    summary = pd.DataFrame(rows).set_index("method")

    names = list(preds)
    agree = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            agree.loc[a, b] = (preds[a] == preds[b]).mean()

    all_same = np.all([preds[n] == preds[names[0]] for n in names], axis=0)

    return summary, agree.round(3), all_same, preds

summary, agree, all_same, preds = diagnose(scores_dict)

print(summary.to_string(), "\n")
print("Pairwise agreement of top-1 document:\n", agree.to_string(), "\n")
print(f"All methods agree on {all_same.mean():.1%} of questions")

disagree = pd.DataFrame(preds)
print("\nQuestions where methods disagree:")
print(disagree[~all_same].head(20))

                      shape  nan_count  unique_docs_predicted  top_doc  top_doc_share  flat_rows  mean_margin
method                                                                                                       
tf-idf           (400, 137)       4000                     56        8          0.198          0          NaN
bm25             (400, 137)       4000                     56        8          0.150          0          NaN
cosine_small     (400, 137)       4000                     62       66          0.102          0          NaN
cosine_large     (400, 137)       4000                     59       70          0.112          0          NaN
euclidean_small  (400, 137)       4000                     51      117          0.285          0          NaN
euclidean_large  (400, 137)       4000                     59       70          0.108          0          NaN 

Pairwise agreement of top-1 document:
                  tf-idf   bm25  cosine_small  cosine_large  euclidean_small  eu

/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),
/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),
/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),
/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),
/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),
/tmp/ipykernel_13133/1171317935.py:36: RuntimeWarning: Mean of empty slice
  "mean_margin": round(float(np.nanmean(margin)), 4),


## Conclusion

We compared six retrieval methods (tf-idf, BM25, and cosine/Euclidean similarity on two different embedding models) over 400 questions and 137 documents. The score matrices were checked for whether all questions collapse onto a single document.

**No method collapses onto one document.** Each method retrieves between 51 and 62 distinct documents as top-1.

**Hub documents exist in every method.** One document attracts a disproportionate share of questions: 20% for tf-idf, 15% for BM25, 10-11% for the cosine methods, and 28.5% for `euclidean_small`. Without ground-truth labels we cannot tell whether this is legitimate (many questions may come from the same document) or a retrieval bias.

**The methods disagree substantially.** All six agree on only 11.2% of questions. tf-idf and BM25 agree with each other (71%), and the two large-model variants agree with each other (83%), but lexical and embedding methods overlap little (18-34%). Disagreement alone does not show which method is best; that requires accuracy@1 or MRR against the correct documents.

**Euclidean distance is not useful here.** The models are trained for cosine similarity, and for normalised embeddings Euclidean distance gives exactly the same ranking as cosine, so it adds no information. Our Euclidean and cosine results do not match (55% agreement for the small model, 83% for the large), which indicates the embeddings are not normalised and vector length is distorting the distances. The clearest symptom is `euclidean_small`, where a single document attracts 28.5% of all questions. We therefore use cosine similarity. Euclidean distance will still be held as an option.

**Next steps**
2. Evaluate against ground truth (accuracy@1, MRR) to rank the methods.
3. Once we can evaluate, we should also compare what happens when using larger chunks with models that can handle more tokens.